# Getting Started: Comparing VQE, VarQITE, and QPE on H2

This notebook compares the three main workflows in the repository on the same
small molecule: **H2**.

Goals:

- run a minimal **VQE** calculation
- run a minimal **VarQITE** calculation
- run a minimal **QPE** calculation
- compare their final energy estimates against the exact spectrum

This is a compact comparison notebook rather than a benchmarking notebook.

In [ ]:
from __future__ import annotations

import numpy as np
import matplotlib.pyplot as plt

from common.hamiltonian import get_exact_spectrum
from qite.core import run_qite
from qpe.core import run_qpe
from vqe.core import run_vqe

## Exact reference spectrum

For `H2`, we can compute the exact spectrum and use it as a common reference
for all three methods.

In [ ]:
exact_spectrum = np.asarray(get_exact_spectrum("H2"), dtype=float)
exact_spectrum = np.sort(exact_spectrum)

exact_ground_energy = float(exact_spectrum[0])

print("Exact spectrum:")
print(exact_spectrum)
print()
print(f"Exact ground-state energy: {exact_ground_energy:.10f}")

## Small helper utilities

The three workflows return different result dictionaries, so we use a few
small helpers to keep the notebook compact.

In [ ]:
def first_present(d: dict, candidates: list[str]):
    for key in candidates:
        if key in d:
            return d[key]
    return None


def as_1d_float_array(x):
    if x is None:
        return None
    arr = np.asarray(x, dtype=float)
    return arr.ravel()


def qpe_energy_estimate(res: dict):
    probabilities = as_1d_float_array(
        first_present(res, ["probabilities", "probs", "phase_probabilities"])
    )
    energies = as_1d_float_array(
        first_present(res, ["energies", "energy_grid", "energy_values"])
    )

    if probabilities is not None and energies is not None and len(probabilities) == len(energies):
        return float(energies[int(np.argmax(probabilities))])

    direct = first_present(
        res,
        [
            "best_energy",
            "estimated_energy",
            "energy_estimate",
            "ground_energy_estimate",
            "dominant_energy",
        ],
    )
    if direct is not None:
        return float(direct)

    return None

## Run VQE

We start with a small VQE calculation using a standard ansatz and optimizer.

In [ ]:
vqe_res = run_vqe(
    molecule="H2",
    ansatz_name="UCCSD",
    optimizer_name="Adam",
    steps=50,
    stepsize=0.2,
    seed=0,
    noisy=False,
    force=True,
    plot=False,
)

In [ ]:
vqe_energies = as_1d_float_array(vqe_res["energies"])
vqe_final_energy = float(vqe_res["energy"])

print(f"VQE final energy: {vqe_final_energy:.10f}")

## Run VarQITE

Next we run a small imaginary-time calculation on the same molecule.

In [ ]:
qite_res = run_qite(
    molecule="H2",
    ansatz_name="UCCSD",
    steps=50,
    dtau=0.2,
    seed=0,
    noisy=False,
    force=True,
    plot=False,
)

In [ ]:
qite_energies = as_1d_float_array(qite_res["energies"])
qite_final_energy = float(qite_res["energy"])

print(f"VarQITE final energy: {qite_final_energy:.10f}")

## Run QPE

Finally we run a small QPE calculation. This is a compact demonstration, so
the settings are intentionally modest.

In [ ]:
qpe_res = run_qpe(
    molecule="H2",
    n_ancilla=6,
    t=1.0,
    seed=0,
    noisy=False,
    force=True,
    plot=False,
)

In [ ]:
qpe_final_energy = qpe_energy_estimate(qpe_res)
qpe_final_energy

## Compare final energy estimates

In [ ]:
rows = [
    ("VQE", vqe_final_energy),
    ("VarQITE", qite_final_energy),
    ("QPE", qpe_final_energy),
]

print(f"{'Method':<10} {'Energy':>16} {'|ΔE| vs exact':>18}")
print("-" * 46)
for name, energy in rows:
    if energy is None:
        print(f"{name:<10} {'n/a':>16} {'n/a':>18}")
    else:
        err = abs(float(energy) - exact_ground_energy)
        print(f"{name:<10} {float(energy):>16.10f} {err:>18.6e}")

## Optimization / evolution traces

VQE and VarQITE both return an energy trajectory over iterations, so we can
compare how the two approaches move toward the ground state.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(np.arange(len(vqe_energies)), vqe_energies, marker="o", label="VQE")
plt.plot(np.arange(len(qite_energies)), qite_energies, marker="o", label="VarQITE")
plt.axhline(exact_ground_energy, linestyle="--", label="Exact ground energy")
plt.xlabel("Iteration")
plt.ylabel("Energy [Ha]")
plt.title("VQE vs VarQITE for H2")
plt.grid(True)
plt.legend()
plt.show()

## QPE returned distribution

When available, the QPE distribution is worth inspecting directly.

In [ ]:
qpe_probabilities = as_1d_float_array(
    first_present(qpe_res, ["probabilities", "probs", "phase_probabilities"])
)
qpe_energies = as_1d_float_array(
    first_present(qpe_res, ["energies", "energy_grid", "energy_values"])
)

if qpe_probabilities is not None:
    if qpe_energies is not None and len(qpe_energies) == len(qpe_probabilities):
        x = qpe_energies
        xlabel = "Energy [Ha]"
        title = "QPE energy distribution for H2"
    else:
        x = np.arange(len(qpe_probabilities))
        xlabel = "Index"
        title = "QPE returned distribution for H2"

    plt.figure(figsize=(8, 4))
    plt.bar(
        x,
        qpe_probabilities,
        width=(x[1] - x[0]) if len(x) > 1 and np.issubdtype(np.asarray(x).dtype, np.number) else 0.8,
    )
    plt.xlabel(xlabel)
    plt.ylabel("Probability")
    plt.title(title)
    plt.grid(True)
    plt.show()
else:
    print("No QPE probability distribution was returned.")

## Interpretation

These three methods attack the same spectral problem in different ways:

- **VQE** directly minimizes an energy expectation value
- **VarQITE** follows an imaginary-time projected update
- **QPE** extracts spectral information from controlled time evolution

On a small system like `H2`, all three can be compared cleanly against the
same exact reference spectrum.

## What this notebook showed

We:

- ran minimal `VQE`, `VarQITE`, and `QPE` workflows on `H2`
- compared their returned energy estimates
- compared `VQE` and `VarQITE` energy trajectories
- inspected the returned `QPE` distribution when available

This is the smallest side-by-side comparison notebook in the repository.

## Next steps

Good follow-ups are:

- repeat the comparison for `LiH` or another small molecule
- vary the QPE settings `t` and `n_ancilla`
- compare different ansätze for VQE and VarQITE
- build a larger notebook focused on method trade-offs